In [0]:
CREATE OR REFRESH STREAMING TABLE techdados.bronze.dlt_customers
(
CONSTRAINT valid_customer_id EXPECT (customer_id IS NOT NULL)
)
COMMENT "Raw orders from cloud storage"
AS
SELECT 
    *,
    current_timestamp() AS ingestion_time,
    _metadata.file_path AS source_file
FROM STREAM cloud_files(
    '/Volumes/techdados/bronze/vol_landing/dlt_customer',
    'json',
    map(
        'multiLine', 'true',
        'pathGlobFilter', '*.json'
    )
);

In [0]:
CREATE OR REFRESH STREAMING LIVE TABLE techdados.silver.dlt_customers_base
(
  CONSTRAINT valid_customer_id EXPECT (customer_id IS NOT NULL) ON VIOLATION FAIL UPDATE,
  CONSTRAINT valid_email EXPECT (email IS NOT NULL AND email LIKE '%@%.%') ON VIOLATION DROP ROW
)
COMMENT 'Silver – customers base'
TBLPROPERTIES ("quality" = "silver")
AS
SELECT
    customer_id,
    name,
    email,
    CAST(created_at AS DATE) AS created_at,
    ingestion_time,
    source_file
FROM STREAM(LIVE.dlt_customers)

In [0]:
-- %python
-- import dlt
-- from pyspark.sql.functions import col, explode, from_json
-- from pyspark.sql.types import StructType, StructField, StringType, ArrayType

-- @dlt.table(
--     name="dlt_customer_addresses",
--     comment="Silver – exploded customer addresses with quality checks",
-- )
-- @dlt.expect_or_drop("valid_customer_id", "customer_id IS NOT NULL")
-- @dlt.expect_or_drop("valid_address_type", "address_type IN ('home', 'work', 'delivery', 'other')")
-- @dlt.expect_or_drop("valid_state", "LENGTH(state) = 2")
-- def dlt_customer_addresses():
--     address_schema = ArrayType(
--         StructType([
--             StructField("type", StringType(), True),
--             StructField("city", StringType(), True),
--             StructField("state", StringType(), True),
--             StructField("zip", StringType(), True)
--         ])
--     )
    
--     return (
--         dlt.read_stream("dlt_customers")
--         .withColumn("address_array", from_json(col("addresses"), address_schema))
--         .withColumn("address", explode(col("address_array")))
--         .select(
--             col("customer_id"),
--             col("address.type").alias("address_type"),
--             col("address.city").alias("city"),
--             col("address.state").alias("state"),
--             col("address.zip").alias("zip"),
--             col("ingestion_time")
--         )
--     )

In [0]:
CREATE OR REFRESH STREAMING LIVE TABLE techdados.silver.dlt_customer_addresses
(
  CONSTRAINT valid_customer_id EXPECT (customer_id IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT valid_address_type EXPECT (address_type IN ('home', 'work', 'delivery', 'other')) ON VIOLATION DROP ROW,
  CONSTRAINT valid_state EXPECT (LENGTH(state) = 2) ON VIOLATION DROP ROW
)
COMMENT 'Silver – exploded customer addresses with quality checks'
AS
SELECT
    customer_id,
    addr.col.type   AS address_type,
    addr.col.city   AS city,
    addr.col.state  AS state,
    addr.col.zip    AS zip,
    ingestion_time
FROM STREAM(LIVE.dlt_customers)
LATERAL VIEW explode(from_json(addresses, 'array<struct<type:string,city:string,state:string,zip:string>>')) addr

In [0]:
CREATE OR REFRESH STREAMING TABLE dlt_customers_scd2;
APPLY CHANGES INTO
  LIVE.dlt_customers_scd2
FROM
  (
    SELECT 
      CAST(customer_id AS INT) AS id_cliente,
      * EXCEPT (_rescued_data,customer_id)
    FROM STREAM(LIVE.dlt_customers)
    WHERE customer_id IS NOT NULL
  )
KEYS
  (id_cliente)
SEQUENCE BY
  created_at
STORED AS
  SCD TYPE 2;

In [0]:
select * from techdados.bronze.dlt_customers_scd2
where __END_AT is null